# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Traducción de la imagen (fila, columna)
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}       # estanterías
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}     # piso resbaloso

        self.terminal_states = {
            (0, 5): +10.0,   # zona de entrega
            (2, 2): +2.0,    # estación de carga
            (3, 5): -10.0,   # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        # Dinámica: (P(dirección elegida), P(cada desviación lateral))
        self.p_normal = (0.90, 0.05)
        self.p_slippery = (0.60, 0.20)

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s): depende únicamente del estado actual.
        if self.is_terminal(state):
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        # Los terminales son absorbentes.
        if self.is_terminal(state):
            return [(state, 1.0)]

        # T(s,a,s') depende del tipo de piso del estado actual.
        if state in self.slippery_states:
            p_intended, p_side = self.p_slippery
        else:
            p_intended, p_side = self.p_normal

        # Desviaciones relativas a la dirección elegida (dr, dc):
        # izquierda = rotar 90° antihorario, derecha = rotar 90° horario.
        left = (-action[1], action[0])
        right = (action[1], -action[0])

        outcomes = [
            (action, p_intended),
            (left, p_side),
            (right, p_side),
        ]

        transitions = []
        for move, prob in outcomes:
            next_state = (state[0] + move[0], state[1] + move[1])

            # Si sale del grid o golpea una estantería: s' = s.
            if not self.is_valid_state(next_state):
                next_state = state

            transitions.append((next_state, prob))

        return transitions


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    # V_0(s) = 0 para todos los estados
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        # Actualización sincrónica: V_{k+1} se calcula solo con V_k
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                # max_a sum_{s'} T(s,a,s') V_k(s')
                best_expected_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )
                # V_{k+1}(s) = R(s) + gamma * max_a sum T V_k
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * best_expected_value
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state])
            )

        V = V_new

        if biggest_change < threshold:
            break

    return V, iteration + 1


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # V_{k+1}^pi(s) = R(s) + gamma sum_{s'} T(s,pi(s),s') V_k^pi(s')
    # Aquí NO hay max: la acción la fija policy[state].
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                action = policy[state]
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(grid, state, action, V)
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state])
            )

        V = V_new

        if biggest_change < threshold:
            break

    return V, iteration + 1


def policy_improvement(grid, V):
    # pi_new(s) = argmax_a sum_{s'} T(s,a,s') V^pi(s')
    new_policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        new_policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. política inicial arbitraria: UP en todos los estados no terminales
    policy = {
        state: (-1, 0)
        for state in grid.states()
        if not grid.is_terminal(state)
    }

    history = []

    for iteration in range(max_iter):
        # 2. evaluación
        V, eval_iterations = policy_evaluation(grid, policy, threshold=threshold)

        # 3. mejora
        new_policy = policy_improvement(grid, V)

        changed = sum(
            new_policy[state] != policy[state]
            for state in new_policy
        )

        history.append({
            "policy_iteration": iteration + 1,
            "evaluation_sweeps": eval_iterations,
            "changed_actions": changed,
        })

        policy = new_policy

        # 4. repetir hasta estabilidad
        if changed == 0:
            break

    V, _ = policy_evaluation(grid, policy, threshold=threshold)

    return policy, V, history


## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [{'policy_iteration': 1, 'evaluation_sweeps': 89, 'changed_actions': 16}, {'policy_iteration': 2, 'evaluation_sweeps': 77, 'changed_actions': 5}, {'policy_iteration': 3, 'evaluation_sweeps': 22, 'changed_actions': 1}, {'policy_iteration': 4, 'evaluation_sweeps': 22, 'changed_actions': 0}]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


### Respuestas — Parte 5

**1. Desde `START`, ¿busca la entrega +10 o la carga +2?**

Con los parámetros base ($R=-1$, $\gamma=0.9$) el robot **prefiere la estación de carga +2**. La trayectoria (si no hay desviaciones) es:

`(0,0) → (0,1) → (0,2) → (1,2) → (2,2)` = carga +2.

El punto de decisión clave es `(1,2)`: allí la política elige ↓ (carga) en lugar de → (camino a la entrega).

**2. ¿Por qué una recompensa menor podría ser óptima?**

Porque el agente no maximiza la recompensa terminal, sino el **retorno esperado descontado**. Para llegar a +10 desde `(1,2)` hacen falta 4 pasos más (`(1,3)→(1,4)→(1,5)→(0,5)`), cada uno con costo $-1$, y **obligatoriamente** hay que pasar por el peligro `(1,4)` = $-3$ (la entrega `(0,5)` solo es alcanzable a través de `(0,4)` o `(1,5)`, y ambas solo se alcanzan desde `(1,4)`). Además, el +10 queda descontado por $\gamma^4 \approx 0.66$ y existe riesgo de desviarse hacia el peligro mortal $-10$ en `(3,5)`. La carga, en cambio, está a un paso: $+2$ seguro y cercano vale más que $+10$ lejano, costoso y arriesgado.

**3. ¿En qué estados el piso resbaloso cambia la decisión?**

Comparando con el mismo almacén pero con todo el piso normal (0.9/0.05/0.05), la política cambia en **`(1,2)`** y **`(3,1)`**:

- `(1,2)`: sin resbalar elige → (ruta a +10); con piso resbaloso elige ↓ (carga). Ir a la derecha desde un estado resbaloso tiene un 40 % de desviarse (hacia arriba o hacia abajo), mientras que ↓ hacia la carga es "seguro": incluso las desviaciones laterales llevan a `(1,3)` o la dejan en `(1,2)`. Este cambio es justamente el que hace que el robot prefiera la carga desde `START`.
- `(3,1)`: sin resbalar sube por `(2,1)`; con resbaloso prefiere → hacia `(3,2)`, evitando pasar por la celda resbalosa `(2,1)`: desde allí, intentar → hacia la carga falla con probabilidad 0.4 (choca con la estantería `(1,1)` o cae de vuelta a `(3,1)`), mientras que `(3,2)` es piso normal y desde ella ↑ llega a la carga con probabilidad 0.9.

**4. ¿Qué papel cumple el costo por paso `-1`?**

Es una "penalización por tiempo": cada paso adicional resta 1, así que el agente quiere terminar rápido. Esto favorece terminales cercanos (la carga +2) frente a lejanos (+10) y castiga los rodeos. Con un costo grande el agente es "impaciente"; con un costo pequeño puede permitirse caminos largos hacia la recompensa grande (ver Experimento A y Bonus).

**5. ¿Por qué $T(s,a,s')$ ya no puede usar las mismas probabilidades en todos los estados?**

Porque la dinámica depende del **tipo de piso del estado actual**: en piso normal $P(\text{deseada})=0.9$ y en piso resbaloso $P(\text{deseada})=0.6$. Por eso `get_transition_probs(state, action)` primero consulta si `state` está en `slippery_states` y solo después arma la distribución. $T$ sigue siendo markoviana (solo depende de $s$ y $a$), pero ya no es homogénea en el espacio.


In [7]:
# Herramientas para los experimentos

def rollout_intended(grid, policy, max_steps=30):
    """Camino que sigue la política si cada acción sale como se desea."""
    s = grid.start
    path = [s]
    while not grid.is_terminal(s) and len(path) < max_steps:
        a = policy[s]
        nxt = (s[0] + a[0], s[1] + a[1])
        s = nxt if grid.is_valid_state(nxt) else s
        path.append(s)
    return path


def run_experiment(title, **changes):
    g = WarehouseMDP()
    for name, value in changes.items():
        setattr(g, name, value)

    V, n = value_iteration(g)
    pi = extract_policy(g, V)

    print(f"=== {title} ===  cambios: {changes}")
    print("Iteraciones VI:", n)
    print_values(g, V)
    print()
    print_policy(g, pi)

    path = rollout_intended(g, pi)
    print("\nCamino desde START:", " → ".join(map(str, path)))
    print(f"Termina en {path[-1]} (R = {g.get_reward(path[-1]):+.0f})")

    changed = [s for s in pi_vi if pi_vi[s] != pi[s]]
    print("Estados donde cambia la política respecto a la base:", changed)
    return g, V, pi


# Caso base, como referencia
_ = run_experiment("BASE")

=== BASE ===  cambios: {}
Iteraciones VI: 20
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Camino desde START: (0, 0) → (0, 1) → (0, 2) → (1, 2) → (2, 2)
Termina en (2, 2) (R = +2)
Estados donde cambia la política respecto a la base: []


### Experimento A — Menos costo por paso (`living_reward = -0.1`)

**Predicción (antes de ejecutar):** al ser más barato caminar, el rodeo hacia +10 ya no es tan costoso; esperamos que desde `START` el robot cambie de la carga +2 a la entrega +10.

In [8]:
g_A, V_A, pi_A = run_experiment("Experimento A", living_reward=-0.1)

=== Experimento A ===  cambios: {'living_reward': -0.1}
Iteraciones VI: 26
 +1.649 |  +1.992 |  +2.361 |   WALL   |  +8.591 | +10.000
 +1.358 |   WALL   |  +2.797 |  +3.911 |  +4.550 |  +8.591
 +1.259 |  +1.533 |  +2.000 |  +3.306 |   WALL   |  +7.537
 +1.243 |  +1.540 |  +2.040 |  +2.417 |  +2.026 | -10.000
 +0.865 |  -1.794 |   WALL   |  +2.026 |  +1.709 |  +0.874

 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Camino desde START: (0, 0) → (0, 1) → (0, 2) → (1, 2) → (1, 3) → (1, 4) → (1, 5) → (0, 5)
Termina en (0, 5) (R = +10)
Estados donde cambia la política respecto a la base: [(1, 0), (1, 2), (3, 2)]


**Resultado:** la predicción se cumple. Desde `START` el robot ahora va a la **entrega +10** por `(0,0)→(0,1)→(0,2)→(1,2)→(1,3)→(1,4)→(1,5)→(0,5)`. El cambio decisivo vuelve a estar en `(1,2)` (↓ pasa a →). También cambian `(1,0)` (↓ pasa a ↑, porque ahora conviene volver a la fila 0 para ir hacia la derecha) y `(3,2)` (↑ pasa a →, porque sale a buscar la entrega rodeando por la derecha). Todos los valores suben porque cada paso cuesta menos.

### Experimento B — Piso muy resbaloso (`0.60 → 0.40`, desviaciones `0.30 / 0.30`)

**Predicción (antes de ejecutar):** el piso resbaloso se vuelve aún más peligroso, así que la preferencia por la carga se refuerza y los estados vecinos a celdas resbalosas pierden valor. No esperamos que el robot cambie de objetivo.

In [9]:
g_B, V_B, pi_B = run_experiment("Experimento B", p_slippery=(0.40, 0.30))

=== Experimento B ===  cambios: {'p_slippery': (0.4, 0.3)}
Iteraciones VI: 22
 -2.706 |  -1.809 |  -0.797 |   WALL   |  +7.607 | +10.000
 -2.652 |   WALL   |  +0.395 |  +2.104 |  +3.670 |  +7.607
 -1.744 |  -0.670 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.831 |  -0.774 |  +0.534 |  -1.137 |  -2.152 | -10.000
 -2.785 |  -3.929 |   WALL   |  -2.152 |  -2.974 |  -4.041

 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Camino desde START: (0, 0) → (0, 1) → (0, 2) → (1, 2) → (2, 2)
Termina en (2, 2) (R = +2)
Estados donde cambia la política respecto a la base: []


**Resultado:** la **política no cambia** (el robot sigue yendo a la carga +2), pero los valores bajan en los estados que dependen de celdas resbalosas, por ejemplo $V(1,2)$: $0.560 \to 0.395$ y $V(2,1)$: $-0.063 \to -0.670$. La decisión desde `(1,2)` hacia la carga ya era la más robusta; aumentar el resbalamiento solo la refuerza.

### Experimento C — Más paciencia (`gamma = 0.99`)

**Predicción (antes de ejecutar):** con $\gamma$ cercano a 1 el +10 lejano casi no se descuenta ($0.99^4 \approx 0.96$ frente a $0.9^4 \approx 0.66$), así que esperamos que el robot valore más la entrega.

In [10]:
g_C, V_C, pi_C = run_experiment("Experimento C", gamma=0.99)

=== Experimento C ===  cambios: {'gamma': 0.99}
Iteraciones VI: 24
 -1.456 |  -0.310 |  +0.809 |   WALL   |  +8.601 | +10.000
 -2.179 |   WALL   |  +2.003 |  +4.118 |  +5.354 |  +8.601
 -1.081 |  +0.119 |  +2.000 |  +2.913 |   WALL   |  +7.395
 -1.606 |  -0.467 |  +0.799 |  +0.817 |  -0.359 | -10.000
 -2.752 |  -3.737 |   WALL   |  -0.359 |  -1.408 |  -2.892

 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Camino desde START: (0, 0) → (0, 1) → (0, 2) → (1, 2) → (1, 3) → (1, 4) → (1, 5) → (0, 5)
Termina en (0, 5) (R = +10)
Estados donde cambia la política respecto a la base: [(1, 2)]


**Resultado:** sí, la política **valora más el +10 distante**. El único estado que cambia es `(1,2)` (↓ pasa a →), pero es suficiente para que desde `START` el robot vaya a la **entrega +10**. Los valores de toda la región cercana a la entrega suben (por ejemplo $V(1,3)$: $2.104 \to 4.118$). Aun así, el costo por paso $-1$ sigue presente, por eso los estados lejanos (fila 4) siguen teniendo valores negativos.

### Bonus — Umbral de `living_reward`

Barremos `living_reward` y buscamos el punto donde la política desde `START` cambia de objetivo. Como la decisión se toma en `(1,2)`, refinamos el umbral por bisección mirando la acción en ese estado.

In [11]:
import numpy as np

def goal_from_start(living_reward, threshold=1e-4):
    g = WarehouseMDP()
    g.living_reward = living_reward
    V, _ = value_iteration(g, threshold=threshold)
    pi = extract_policy(g, V)
    return rollout_intended(g, pi)[-1], pi


# 1) Barrido grueso
previous = None
for lr in np.round(np.arange(-2.0, 0.0 + 1e-9, 0.1), 2):
    goal, _ = goal_from_start(float(lr))
    marker = "  <-- cambio" if previous is not None and goal != previous else ""
    print(f"living_reward = {lr:+.2f} -> termina en {goal}{marker}")
    previous = goal


# 2) Bisección fina sobre la acción en (1,2)
def action_at_decision(lr):
    _, pi = goal_from_start(lr, threshold=1e-8)
    return pi[(1, 2)]

lo, hi = -1.0, -0.5   # lo -> carga, hi -> entrega
for _ in range(40):
    mid = (lo + hi) / 2
    if action_at_decision(mid) == action_at_decision(lo):
        lo = mid
    else:
        hi = mid

print(f"\nUmbral aproximado: living_reward ≈ {(lo + hi) / 2:.4f}")

living_reward = -2.00 -> termina en (2, 2)
living_reward = -1.90 -> termina en (2, 2)
living_reward = -1.80 -> termina en (2, 2)
living_reward = -1.70 -> termina en (2, 2)
living_reward = -1.60 -> termina en (2, 2)
living_reward = -1.50 -> termina en (2, 2)
living_reward = -1.40 -> termina en (2, 2)
living_reward = -1.30 -> termina en (2, 2)


living_reward = -1.20 -> termina en (2, 2)
living_reward = -1.10 -> termina en (2, 2)
living_reward = -1.00 -> termina en (2, 2)
living_reward = -0.90 -> termina en (2, 2)
living_reward = -0.80 -> termina en (2, 2)
living_reward = -0.70 -> termina en (0, 5)  <-- cambio
living_reward = -0.60 -> termina en (0, 5)
living_reward = -0.50 -> termina en (0, 5)
living_reward = -0.40 -> termina en (0, 5)
living_reward = -0.30 -> termina en (0, 5)
living_reward = -0.20 -> termina en (0, 5)
living_reward = -0.10 -> termina en (0, 5)
living_reward = +0.00 -> termina en (0, 5)

Umbral aproximado: living_reward ≈ -0.7983


**Conclusión del bonus:** el umbral está en **`living_reward` ≈ −0.80**.

- `living_reward` < −0.80 → el robot va a la **carga +2** (cada paso es demasiado caro para el rodeo).
- `living_reward` > −0.80 → el robot intenta llegar a la **entrega +10**.

Por eso, con el valor original $-1$ el robot elige la carga, y con $-0.1$ (Experimento A) elige la entrega.